## Testing normalizeTextForTts Mutation 🐞🚧

So when I made some changes my integration tests suddenly started to fail for the following input

In [2]:
# P.S. => Postscript or "By the way, one more thing:"
# A-Are => this should sound like a stutter, though IDK how to represent that so Piper can sound like it is stuttering.

CHUNK_A = """
Mireya, Sylvia, and Valeria looked at Yuan with wide eyes, their gazes fixed on the cultivation techniques in his hands.

"A-Are those cultivation techniques…?" Sylvia asked in a surprised tone, her curiosity evident.

"That's right. These are for you. Once you learn them, you can start cultivating," Yuan said with a smile before handing each of them their specific techniques. They accepted them eagerly.

Valeria quickly opened her technique and began to read, but she couldn't understand a single thing. It was as if the content was obscured by a thick mist, making it impossible to decipher.

Immersed in this 'water' Michael heard the fading voice of the annoyed God.

"P.S. I gave you abilities based on your desires. It's unknown for now, but you'll find out about it soon enough.

And since I took you away from your very comfortable life, I have decided to reincarnate you as the youngest son of the richest family in the entire world. So, live a life of luxury, Mr. Stoic."


"""

Basically this was a sudden change of heart for `qwen2.5:3b` in my integration tests. The good news is that our **automated integration** test and guardrail for **checking the text length** proved invaluable in **catching this issue early one** instead of users catching it. So here is the error message in question in the GitHub Actions pipeline:

```shell
______________ test_chunked_long_story_emits_gen_ai_token_metrics ______________

http_client = <httpx.AsyncClient object at 0x7f6f4121cc20>
otel_spans_reader = <function otel_spans_reader.<locals>._read at 0x7f6f4123c2c0>

    @pytest.mark.integration
    async def test_chunked_long_story_emits_gen_ai_token_metrics(
        http_client: AsyncClient,
        otel_spans_reader: SpansReader,
    ) -> None:
        # Arrange
        user_id = "test-user-long-story"
        normalised_chunks: list[str] = []
        chunk_count = len(LONG_STORY_CHUNKS)
    
        # Act
        for index, chunk in enumerate(LONG_STORY_CHUNKS):
            response = await http_client.post(
                "/graphql",
                headers={"x-app-user-id": user_id},
                json={
                    "query": NORMALIZE_TTS_MUTATION,
                    "variables": {"text": chunk},
                },
            )
            assert response.status_code == 200, (index, response.text)
            body = response.json()
>           assert body.get("errors") is None, (index, body)
E           AssertionError: (0, {'data': None, 'errors': [{'message': 'TTS normalization output length (296) deviates more than 30% from input length (985).', 'locations': [{'line': 3, 'column': 9}], 'path': ['normalizeTextForTts'], 'extensions': {'code': 'LENGTH_DEVIATION', 'inputLength': 985, 'outputLength': 296, 'maxDeviation': 0.3}}]})
E           assert [{'message': 'TTS normalization output length (296) deviates more than 30% from input length (985).', 'locations': [{'line': 3, 'column': 9}], 'path': ['normalizeTextForTts'], 'extensions': {'code': 'LENGTH_DEVIATION', 'inputLength': 985, 'outputLength': 296, 'maxDeviation': 0.3}}] is None
E            +  where [{'message': 'TTS normalization output length (296) deviates more than 30% from input length (985).', 'locations': [{'line': 3, 'column': 9}], 'path': ['normalizeTextForTts'], 'extensions': {'code': 'LENGTH_DEVIATION', 'inputLength': 985, 'outputLength': 296, 'maxDeviation': 0.3}}] = <built-in method get of dict object at 0x7f6f41ba6580>('errors')
E            +    where <built-in method get of dict object at 0x7f6f41ba6580> = {'data': None, 'errors': [{'message': 'TTS normalization output length (296) deviates more than 30% from input length (985).', 'locations': [{'line': 3, 'column': 9}], 'path': ['normalizeTextForTts'], 'extensions': {'code': 'LENGTH_DEVIATION', 'inputLength': 985, 'outputLength': 296, 'maxDeviation': 0.3}}]}.get

tests/test_normalize_tts_graphql.py:103: AssertionError
```

That is why I decided to test this out a bit more interactively in a Jupyter Notebook. Here I will use OpenAI models and see for myself what might be wrong.

In [3]:
from os import getenv
from typing import cast

from dotenv import load_dotenv


load_dotenv(override=True)


openai_api_key = cast(str, getenv("OPENAI_API_KEY"))
openrouter_api_key = cast(str, getenv("OPENROUTER_API_KEY"))
openrouter_url = "https://openrouter.ai/api/v1"

if not openai_api_key:
    raise ValueError("OPENAI_API_KEY is not set in the environment variables.")

if not openrouter_api_key:
    raise ValueError("OPENROUTER_API_KEY is not set in the environment variables.")

## The First Test -- no Structured Output

So Claude Code was suggesting if I drop the whole structured output. I decided to put that into test and it works. But at what cost? Do I really have to give up structured output because the model is dumb? That is why I have a second test coming up.

In [ ]:
from pydantic_ai import Agent
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.openai import OpenAIProvider

from src.modules.normalize_tts import agent as agent_mod
from src.utils import load_prompt


model = OpenAIChatModel("gpt-4o-mini", provider=OpenAIProvider())  # picks up OPENAI_API_KEY
agent = Agent(model, model_settings={"temperature": 0})

prompt = load_prompt(agent_mod._PROMPTS_DIR, agent_mod.PROMPT_VERSION, text=CHUNK_A)
result = await agent.run(prompt)

print(f" --- input {len(CHUNK_A)} -> output {len(result.output)}")
print(result.output)
print()

 --- input 989 -> output 986
Mireya, Sylvia, and Valeria looked at Yuan with wide eyes, their gazes fixed on the cultivation techniques in his hands.

"A-are those cultivation techniques...?" Sylvia asked in a surprised tone, her curiosity evident.

"That's right. These are for you. Once you learn them, you can start cultivating," Yuan said with a smile before handing each of them their specific techniques. They accepted them eagerly.

Valeria quickly opened her technique and began to read, but she couldn't understand a single thing. It was as if the content was obscured by a thick mist, making it impossible to decipher.

Immersed in this water, Michael heard the fading voice of the annoyed God.

"P.S. I gave you abilities based on your desires. It's unknown for now, but you'll find out about it soon enough.

And since I took you away from your very comfortable life, I have decided to reincarnate you as the youngest son of the richest family in the entire world. So, live a life of luxu

## Second Test

Here I will not compromise on structured output.

In [17]:
import os

from src.modules.normalize_tts import agent as agent_mod
from src.utils import Settings, get_settings, load_prompt


get_settings.cache_clear()
os.environ["LLM__BASE_URL"] = "https://api.openai.com/v1"
os.environ["LLM__MODEL"] = "gpt-4o-mini"
os.environ["LLM__API_KEY"] = openai_api_key

openai_agent = agent_mod.build_agent(Settings())
prompt = load_prompt(agent_mod._PROMPTS_DIR, agent_mod.PROMPT_VERSION, text=CHUNK_A)
result = await openai_agent.run(prompt)

print(f"--- input {len(CHUNK_A)} -> output {len(result.output.normalized_text)}")
print(result.output.normalized_text)

--- input 989 -> output 986
Mireya, Sylvia, and Valeria looked at Yuan with wide eyes, their gazes fixed on the cultivation techniques in his hands.

"A-are those cultivation techniques...?" Sylvia asked in a surprised tone, her curiosity evident.

"That's right. These are for you. Once you learn them, you can start cultivating," Yuan said with a smile before handing each of them their specific techniques. They accepted them eagerly.

Valeria quickly opened her technique and began to read, but she couldn't understand a single thing. It was as if the content was obscured by a thick mist, making it impossible to decipher.

Immersed in this water, Michael heard the fading voice of the annoyed God.

"P.S. I gave you abilities based on your desires. It's unknown for now, but you'll find out about it soon enough.

And since I took you away from your very comfortable life, I have decided to reincarnate you as the youngest son of the richest family in the entire world. So, live a life of luxur

| Model       | Output Mode | Result    | Response Time |
| ----------- | ----------- | --------- | ------------- |
| gpt-4o-mini | Plain       | ✅ Passed | ~3 seconds    |
| gpt-4o-mini | Structured  | ✅ Passed | ~3 seconds    |

## Caution 🚨

I initially thought `qwen2.5:3b` is too small and unintelligent for this task. But what I tested so far was just confirming that a larger frontier model won't get confused and will be able to escape those double quotes nicely for us.

I am still not sure why this became an issue after I added the two new `generateAudio` and `audioVoices`. Maybe it was there and just waiting to show its fangs 🤔? So next I am gonna test it against larger models from the same Qwen family. I will use OpenRouter to keep this simple and easy. For this reason you will be needing their API keys too.

### Tests -- Structured Output

In [ ]:
import asyncio

from src.modules.normalize_tts import agent as agent_mod
from src.utils import Llm, Settings, load_prompt


MODELS = [
    "qwen/qwen-2.5-7b-instruct",
    "qwen/qwen3-8b",
    "qwen/qwen3-14b",
]
prompt = load_prompt(agent_mod._PROMPTS_DIR, agent_mod.PROMPT_VERSION, text=CHUNK_A)


async def run_one(model_name: str) -> tuple[str, str]:
    settings = Settings(
        llm=Llm(
            base_url=openrouter_url,
            model=model_name,
            api_key=openrouter_api_key,
        )
    )
    openai_agent = agent_mod.build_agent(settings)
    result = await openai_agent.run(prompt)
    return model_name, result.output.normalized_text


results = await asyncio.gather(*(run_one(name) for name in MODELS))

for model_name, output in results:
    print()
    print()
    print(f"--- {model_name} --- input {len(CHUNK_A)} -> output {len(output)}")
    print(output)
    print()
    print()



--- qwen/qwen3-8b --- input 989 -> output 988
Mireya, Sylvia, and Valeria looked at Yuan with wide eyes, their gazes fixed on the cultivation techniques in his hands.

"a... are those cultivation techniques…?" Sylvia asked in a surprised tone, her curiosity evident.

"That's right. These are for you. Once you learn them, you can start cultivating," Yuan said with a smile before handing each of them their specific techniques. They accepted them eagerly.

Valeria quickly opened her technique and began to read, but she couldn't understand a single thing. It was as if the content was obscured by a thick mist, making it impossible to decipher.

Immersed in this 'water' Michael heard the fading voice of the annoyed God.

"p.s. I gave you abilities based on your desires. It's unknown for now, but you'll find out about it soon enough.

And since I took you away from your very comfortable life, I have decided to reincarnate you as the youngest son of the richest family in the entire world. So

### Tests -- Plain Output

In [ ]:
import asyncio

from pydantic_ai import Agent
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.openai import OpenAIProvider

from src.modules.normalize_tts import agent as agent_mod
from src.utils import load_prompt


provider = OpenAIProvider(base_url=openrouter_url, api_key=openrouter_api_key)

MODELS = [
    "qwen/qwen-2.5-7b-instruct",
]

prompt = load_prompt(agent_mod._PROMPTS_DIR, agent_mod.PROMPT_VERSION, text=CHUNK_A)


async def run_one(model_name: str) -> tuple[str, str]:
    model = OpenAIChatModel(model_name, provider=provider)
    agent = Agent(model, model_settings={"temperature": 0})
    result = await agent.run(prompt)
    return model_name, result.output


results = await asyncio.gather(*(run_one(name) for name in MODELS))

for model_name, output in results:
    print(f"--- {model_name} --- input {len(CHUNK_A)} -> output {len(output)}")
    print(output)
    print()

--- qwen/qwen-2.5-7b-instruct --- input 989 -> output 996
"""

mireya, sylvia, and valeria looked at yuan with wide eyes, their gazes fixed on the cultivation techniques in his hands.

"a-are those cultivation techniques…?" sylvia asked in a surprised tone, her curiosity evident.

"that's right. these are for you. once you learn them, you can start cultivating," yuan said with a smile before handing each of them their specific techniques. they accepted them eagerly.

valeria quickly opened her technique and began to read, but she couldn't understand a single thing. it was as if the content was obscured by a thick mist, making it impossible to decipher.

immersed in this 'water', michael heard the fading voice of the annoyed god.

"p.s. i gave you abilities based on your desires. it's unknown for now, but you'll find out about it soon enough.

and since i took you away from your very comfortable life, i have decided to reincarnate you as the youngest son of the richest family in the ent

## Results

| Model                     | Output Mode | Result    | Response time |
| ------------------------- | -----       | --------- | ------------- |
| qwen/qwen-2.5-7b-instruct | Plain       | ✅ Passed | ~3 Seconds    |
| qwen/qwen-2.5-7b-instruct | Structured  | ❌ Failed | ~1 Seconds    |
| qwen/qwen3-8b             | Structured  | ✅ Passed | ~1.10 Seconds |
| qwen/qwen3-14b            | Structured  | ✅ Passed | ~27 Seconds   |

Since my test with 7B parameters for plain output mode passed I did not even bother testing further. But I guess then the same should be true for `qwen2.5:3b`. Have not verified it though.

`qwen/qwen3-8b` was extremely slow. I suspect it might be related to [Alibaba's servers](https://openrouter.ai/provider/alibaba) 🤔.

## Conclusion

The bigger the model the better the results. But at the same time it took much longer to respond, but keep in mind that Qwen3 14B in OpenRouter is a reasoning model. So I guess for sanatizing text before sending it to a TTS this is invaluable and let's not forget that it is way cheaper than GPT-4o-mini. You can find the comparison I between the two here: https://openrouter.ai/compare/qwen/qwen3-14b/openai/gpt-4o-mini

## Claude's Follow-up 🤖

> Everything below this point is Claude's own investigation and conclusion, not Kasir's. Recorded here for future reference since it directly informed the fix that shipped.

Kasir asked whether the fix was to try another provider, or to eval-sweep more models before deciding. I tested directly against the **actual CI model** (`qwen2.5:3b` via the local Ollama container, same image the pipeline builds) instead of only the larger OpenRouter models this notebook covers, and found the OpenRouter results didn't transfer:

**1. Reproduced the CI failure exactly.** Running `CHUNK_A` through `build_agent()` with the existing `NativeOutput(NormalizedText)` truncated at 296 characters — same number as the CI failure — and the cutoff landed right before the first embedded straight double-quote in the dialogue (`"That's right...`). This confirms the notebook's escaping theory above: with grammar-constrained JSON decoding (Ollama v0.5.0+), the model doesn't know it must escape a literal `"` inside the JSON string value, so it treats it as the string's closing quote and stops there.

**2. Tried dropping structured output entirely (`output_type=str`).** This is the fix a plain reading of the notebook's Results table above suggests ("plain output passed for `qwen-2.5-7b-instruct`"). Against the real `qwen2.5:3b`, it does NOT truncate — but it deterministically **duplicates the entire input verbatim** (988 → 1974 chars, exactly two copies), reproducible across repeated runs and independent of temperature. Losing the JSON-forced stop condition apparently lets this small model run past its answer and echo the input as a second "turn." So the notebook's conclusion above ("give up structured output") does not hold for the model actually used in CI — it only happened to work for the bigger models tested here.

**3. Tried `ToolOutput` (function-calling) instead of `NativeOutput`.** Failed outright — `UnexpectedModelBehavior: Exceeded maximum output retries (1)`. Worse than both of the above for this model.

**4. The fix that actually worked:** keep `NativeOutput(NormalizedText)` (so the JSON schema still gives a hard stop condition), and add one explicit rule to `prompts/v1.jinja2`:

> "Your output is placed into a JSON string field. Replace every literal double-quote character (\") in the text with a single quote ('); do not try to escape it with a backslash."

This sidesteps the escaping problem instead of asking a 3B model to get JSON string-escaping right — TTS doesn't voice punctuation, so swapping quote style loses nothing. Verified against all three chunks in `tests/fixtures/normalize_tts_graphql_fixture.py` (`LONG_STORY_CHUNKS`) directly through `normalize_tts_via_agent`, deterministically, across repeated runs:

| Chunk | Input len | Output len | Deviation |
| ----- | --------- | ---------- | --------- |
| A     | 989       | 972        | 1.72%     |
| B     | 800       | 782        | 2.25%     |
| C     | 1427      | 1403       | 1.68%     |

All comfortably under the 30% `LENGTH_DEVIATION` threshold, with no truncation and no repetition.

**Conclusion:** this was never a "model too small / wrong provider" problem — it was a structured-output JSON-escaping problem specific to small local models, and it's fixable in the prompt without changing model, provider, or giving up the schema. No need to eval-sweep a matrix of models or download bigger ones; a one-line prompt rule plus the existing local model resolved it.

(Note: I wasn't able to get the full Testcontainers-driven `make integration_test` run to finish in my sandbox — it hit an OOM kill (exit 137), which looks like a sandbox resource limit rather than anything related to this fix. Worth re-running in CI or on a normal dev machine to get the official green check.)
